# Community Notes 陣営反応分析 — Simple 版 (CS授業用)

上から順にセルを実行してください。
「★ 設定」セルのパラメータだけ編集すれば他は変更不要です。

## Simple 版の特徴
- チャンクではなく **noteId サンプリング** で 1 ショット実行（数分〜15 分）
- quality は LLM ラベル学習済みの重みをハードコード（透明）
- 回帰は `type_a + type_b + quality + log_ratings_count` の 4 変数
- trend は目的変数と同じ生データから作られる bad control なので削除

## 仮説判定
- **TypeA のみ有意** → 陣営反応が主因（仮説支持）
- **TypeA・B 両方有意** → 自然拡散も寄与（仮説部分支持）
- **どちらも非有意** → 別要因、またはサンプル不足

## 実行時間の目安
| SAMPLE_FRAC | 所要時間 | 用途 |
|---|---|---|
| 0.05 | 3〜5 分 | 動作確認 |
| 0.30 | 10〜15 分 | 本番 |
| 1.00 | 30〜60 分 | フル（メモリ注意）|

---
## ★ 設定（ここを編集して実行を調整）

In [ ]:
# ====================================================
# ★ サンプリング設定
# ====================================================
# noteId を frac 割でランダムサンプリングし、そのノートに関する
# ratings / history / polarity / bursts を全部まとめて処理する。
# チャンク実行ではないので 1 回走らせれば結果 CSV が一式そろう。
#
# 頑健性チェック: SEED を 1, 2, 3 と変えて同じ frac で再実行し、
#   β_typeA の符号・有意性がブレないかを見る。

SAMPLE_FRAC = 0.30    # ← 0.05 で動作確認、0.30 で本番
SEED        = 42

# ====================================================
# ★ データ場所 (Drive)
# ====================================================
# 既存ノートブックと同じ共有ドライブを参照。
# zip と unzip_data のどちらか存在する方を自動で選ぶ。

DRIVE_DATA_ZIP     = '/content/drive/Shareddrives/基礎プロジェクト/data'
DRIVE_DATA_UNZIPPED = '/content/drive/Shareddrives/基礎プロジェクト/unzip_data'

# ====================================================
# ★ 結果の保存先 (MyDrive)
# ====================================================
SAVE_DIR = '/content/drive/MyDrive/toriumi_x3_results_simple'

# ====================================================
# ★ GitHub (変更不要)
# ====================================================
REPO_URL = 'https://github.com/hirototoda/toriumi_x3.git'
REPO_DIR = '/content/toriumi_x3'

---
## 0. セットアップ（以下は変更不要）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

# zip / unzip_data のどちらが使えるか判定
USE_ZIP = os.path.exists(DRIVE_DATA_ZIP)
USE_UNZIPPED = os.path.exists(DRIVE_DATA_UNZIPPED)

if USE_UNZIPPED:
    DRIVE_DATA = DRIVE_DATA_UNZIPPED
    DATA_MODE = 'unzipped'
    print(f'OK: 解凍済み TSV フォルダを使います → {DRIVE_DATA}')
elif USE_ZIP:
    DRIVE_DATA = DRIVE_DATA_ZIP
    DATA_MODE = 'zip'
    print(f'OK: zip フォルダを使います (解凍が必要) → {DRIVE_DATA}')
else:
    DATA_MODE = None
    print(f'ERROR: 以下のどちらも見つかりません')
    print(f'  - {DRIVE_DATA_ZIP}')
    print(f'  - {DRIVE_DATA_UNZIPPED}')
    print()
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        print('利用できる共有ドライブ:')
        for d in os.listdir(sd):
            print(f'  {d}')
    print('MyDrive (上位 20 件):')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]:
        print(f'  {d}')

if DATA_MODE:
    print()
    print('中身:')
    for d in sorted(os.listdir(DRIVE_DATA)):
        print(f'  {d}')

In [ ]:
# GitHub から clone + pull + 依存インストール
!git clone {REPO_URL} {REPO_DIR} 2>/dev/null || echo 'already cloned'
os.chdir(REPO_DIR)
!git pull
!pip install -q statsmodels

In [ ]:
# data/raw/ にデータを準備
# - DATA_MODE='unzipped' → Drive 上の TSV をそのままコピー（高速）
# - DATA_MODE='zip'      → zip をここで解凍
# どちらも既に raw_dir にあればスキップするので再実行安全。

import glob, zipfile

raw_dir = f'{REPO_DIR}/data/raw'
os.makedirs(raw_dir, exist_ok=True)

def _copy_tsvs(src_folder, prefix):
    if not os.path.exists(src_folder):
        print(f'  [skip] {src_folder} が無い')
        return
    tsvs = sorted(f for f in os.listdir(src_folder) if f.startswith(prefix) and f.endswith('.tsv'))
    for f in tsvs:
        dst = os.path.join(raw_dir, f)
        if os.path.exists(dst):
            print(f'  [skip exists] {f}')
            continue
        print(f'  [copy] {f}')
        subprocess.run(['cp', os.path.join(src_folder, f), dst], check=True)

def _unzip_all(src_folder, prefix):
    if not os.path.exists(src_folder):
        print(f'  [skip] {src_folder} が無い')
        return
    zips = sorted(f for f in os.listdir(src_folder) if f.startswith(prefix) and f.endswith('.zip'))
    for f in zips:
        tsv_name = f.replace('.zip', '.tsv')
        if os.path.exists(os.path.join(raw_dir, tsv_name)):
            print(f'  [skip exists] {tsv_name}')
            continue
        print(f'  [unzip] {f}')
        with zipfile.ZipFile(os.path.join(src_folder, f)) as zf:
            zf.extractall(raw_dir)

for subfolder, prefix in [
    ('RatingData',            'ratings'),
    ('NotesData',             'notes'),
    ('NoteStatusHistoryData', 'noteStatusHistory'),
]:
    print(f'\n● {subfolder} ({prefix}-*)')
    src = os.path.join(DRIVE_DATA, subfolder)
    if DATA_MODE == 'unzipped':
        _copy_tsvs(src, prefix)
    else:
        _unzip_all(src, prefix)

print('\n--- data/raw の中身 ---')
for f in sorted(os.listdir(raw_dir)):
    size_mb = os.path.getsize(os.path.join(raw_dir, f)) / (1024 * 1024)
    print(f'  {f:50s} {size_mb:8.1f} MB')

---
## 1. パイプライン実行

ここが本番。以下を順に実行:
1. notes 全件読 → 政治トピックフィルタ → noteId を `SAMPLE_FRAC` で抽出
2. 該当 ratings / history を読み込み
3. rater の polarity を TruncatedSVD で計算（先頭 50 件で固定、循環論法回避）
4. バースト検出 + TypeA/B 分類
5. quality を LLM 学習済係数で計算
6. ロジスティック回帰 1 本（相関行列 + VIF 診断も自動 print）

実行時間: SAMPLE_FRAC=0.30 で 10〜15 分目安。

In [ ]:
!python scripts/run_simple.py \
    --sample-frac {SAMPLE_FRAC} \
    --seed {SEED}

---
## 2. 結果の確認

In [ ]:
# 2-1. 特徴量テーブル (回帰の入力)
import pandas as pd

feat = pd.read_csv('data/processed/simple_features.csv')
print(f'features: {len(feat):,} notes')
print(f"  deleted (Not Helpful) = {int(feat['deleted'].sum())}")
print(f"  helpful               = {int((feat['deleted']==0).sum())}")
print(f"  typeA bursts          = {int(feat['type_a'].sum())}")
print(f"  typeB bursts          = {int(feat['type_b'].sum())}")
display(feat.head(10))

In [ ]:
# 2-2. バースト一覧
burst_path = 'data/processed/simple_bursts.csv'
if os.path.exists(burst_path):
    bursts = pd.read_csv(burst_path)
    print(f'bursts: {len(bursts):,}')
    print('\nTypeA/B 分布:')
    print(bursts['burst_type'].value_counts(dropna=False))
    display(bursts.head())
else:
    print(f'{burst_path} が無い (バースト検出 0 件)')

In [ ]:
# 2-3. 回帰 summary (テキスト全文)
# type_a の β と p 値を見て仮説を判定する。
print(open('data/processed/simple_regression.txt').read())

---
## 3. 結果を Drive に保存

In [ ]:
# simple_*.csv と simple_*.txt を SAVE_DIR にコピー
import shutil

os.makedirs(SAVE_DIR, exist_ok=True)
copied = 0
for f in os.listdir('data/processed'):
    if f.startswith('simple_'):
        shutil.copy(os.path.join('data/processed', f), SAVE_DIR)
        print(f'  copied: {f}')
        copied += 1
print(f'\nDone! {copied} ファイルを保存: {SAVE_DIR}')

---
## おまけ: 頑健性チェック

同じ `SAMPLE_FRAC` で `SEED` を 1, 2, 3 と変えて 3 回実行し、β_typeA の符号と有意性がブレないか確認する。

1. 上の「★ 設定」セルで `SEED = 1` に変更
2. セル 1 (パイプライン実行) だけ再実行
3. 同様に `SEED = 2`, `SEED = 3` で実行
4. 3 回の結果で β_typeA が一貫して正 かつ 少なくとも 2 回 p < 0.05 なら頑健

※ 出力ファイルは毎回上書きされるので、比較したいときは手動で save dir を分けてください。